In [11]:
!git clone https://github.com/HelloShiwansh/Text-Summarizer

Cloning into 'Text-Summarizer'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 1), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 3.82 MiB | 4.12 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [12]:
%cd Text-Summarizer

/content/Text-Summarizer/Text-Summarizer


In [13]:
import os
print(os.listdir())

['samsum-validation.csv', 'text_summarizer.ipynb', '.gitignore', 'samsum-test.csv', 'samsum-train.csv', '.git']


In [14]:
import torch
print(torch.cuda.is_available())

True


In [15]:
import sys
print(sys.executable)

/usr/bin/python3


In [16]:
!pip install transformers
!pip install "transformers[torch]"

In [17]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [18]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [19]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [20]:
train_data.shape, val_data.shape

((14732, 3), (818, 3))

In [21]:
#random samping
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

## Data Pre-Processing

In [22]:
import re

In [23]:
def clean_data(text):
    text = re.sub(r'\s+', ' ', text)  # Replace multiple whitespace with a single space
    text = re.sub(r'\r\n', ' ', text)  # Replace newlines with a space
    text = re.sub(r"<.*?>", "", text)  # Remove HTML tags
    return text.strip().lower()

In [24]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

## Tokenization

In [25]:
# using t5 tokenizzer to get the t5 vocab and tokenization
tokenizer = T5Tokenizer.from_pretrained('t5-small')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
#raw data -> tokenization -> model input
# for fine tuning

def tokenize(data):
    inputs = tokenizer(data['dialogue'], padding='max_length', truncation=True, max_length=512)
    targets = tokenizer(data['summary'], padding='max_length', truncation=True, max_length=150)

    inputs['labels'] = targets['input_ids']
    return inputs

In [27]:
train_dataset = train_data.apply(tokenize, axis=1).to_list()
val_dataset = val_data.apply(tokenize, axis=1).to_list()

In [28]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [29]:
# input_ids - token ids for the dialogue
# 1 is for EOS token and 0 is for padding

# attention_mask - mask to indicate which tokens are padding and which are not
# length of input_ids and attention_mask is same

# labels - token ids for the summary

In [30]:
len(train_dataset), len(val_dataset)

(4000, 500)

In [31]:
len(train_dataset[0]['input_ids']), len(train_dataset[0]['labels'])

(512, 150)

## Working With the model

In [32]:
# NLP -> generation task which is dependent on the context of the input text -> so conditional generation task

model = T5ForConditionalGeneration.from_pretrained('t5-small')


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [33]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
model.to(device)

Using device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [34]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [36]:
# trainning the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.577381,0.380097
2,0.397485,0.359731
3,0.374499,0.354865
4,0.362143,0.350373
5,0.355740,0.349276
6,0.350639,0.349105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9029809265136719, metrics={'train_runtime': 1450.2115, 'train_samples_per_second': 16.549, 'train_steps_per_second': 2.069, 'total_flos': 3248203235328000.0, 'train_loss': 0.9029809265136719, 'epoch': 6.0})

In [37]:
# model load => fine tunning => Trainning => Saving the model

In [43]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [48]:
!pip install huggingface_hub

from huggingface_hub import notebook_login
notebook_login()

In [49]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("saved_summary_model")
tokenizer = AutoTokenizer.from_pretrained("saved_summary_model")

model.push_to_hub("HelloShiwansh/text-summarizer-model")
tokenizer.push_to_hub("HelloShiwansh/text-summarizer-model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0zi1gka/model.safetensors:  16%|#6        | 39.9MB /  242MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/HelloShiwansh/text-summarizer-model/commit/f14b7e49e0df8227d73cc3020c91f8d70700182f', commit_message='Upload tokenizer', commit_description='', oid='f14b7e49e0df8227d73cc3020c91f8d70700182f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HelloShiwansh/text-summarizer-model', endpoint='https://huggingface.co', repo_type='model', repo_id='HelloShiwansh/text-summarizer-model'), pr_revision=None, pr_num=None)

In [51]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("HelloShiwansh/text-summarizer-model")
tokenizer = AutoTokenizer.from_pretrained("HelloShiwansh/text-summarizer-model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Testing

In [61]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  #tokenise
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding='max_length',
      truncation=True,
      return_tensors='pt'
  ).to(device)

  #generate the summary (token ids)
  model.to(device)
  target = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs['attention_mask'],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  #decoding(ids -> text)
  summary = tokenizer.decode(target[0], skip_special_tokens=True)

  return summary

In [62]:
test_dialogue = """
Journalist: Climate change is becoming one of the most pressing issues worldwide, affecting ecosystems and human lives.

Scientist: Yes, rising global temperatures are causing melting glaciers, rising sea levels, and more extreme weather events.

Journalist: Governments are trying to implement policies, but progress seems slow.

Scientist: That’s true. Transitioning to renewable energy and reducing carbon emissions requires global cooperation and long-term commitment.

Journalist: What role can individuals play in this situation?

Scientist: Individuals can contribute by reducing energy consumption, using public transport, and supporting sustainable products.

Journalist: Are there any positive developments?

Scientist: Absolutely. Advances in solar and wind energy, along with increased awareness, are promising signs.

Journalist: So, there is still hope?

Scientist: Yes, but immediate and consistent action is crucial to mitigate long-term damage.
"""

In [64]:
summary = summarize_dialogue(test_dialogue)
print(summary)

climate change is becoming one of the most pressing issues worldwide, affecting ecosystems and human lives.
